# ✅ 회귀(Regression) 문제 템플릿 v2

## 🍎 이 파일, 언제 쓰나요?
```
정답(Target)이 "연속된 숫자"일 때 사용합니다.

  "이동 시간을 예측"     → 423초 같은 숫자   → ✅ 회귀
  "배송비를 예측"        → 456.5원 같은 숫자 → ✅ 회귀
  "생존 여부를 예측"     → 생존/사망 (범주)  → ❌ 분류 템플릿 사용

판정 기준: 평가지표가 RMSE / MAE / RMSLE / R² 면 회귀입니다.
```

## v2에서 달라진 점 (기존 템플릿 문제점 보완)
| 문제 | v2 개선 |
|---|---|
| 결측치·이상치 처리 셀이 중복/충돌해서 에러가 남 | 각 처리를 **한 번씩만** 순서대로 실행하도록 정리 |
| 어떤 컬럼을 어떻게 처리할지 모르겠음 | `analyze_columns()`가 삭제/OHE/라벨인코딩/스케일링 리스트를 자동 생성 |
| 다른 데이터셋에 적용하면 잘 안 됨 | pandas 버전에 따라 문자열 dtype이 다르게 표시되는 문제 등 호환성 버그 수정 |
| 성능을 더 올리고 싶음 | Step7에 교차검증·앙상블·피처엔지니어링 등 추가 |

## 전체 흐름
```
[Step1] 도구 꺼내기
[Step2] 데이터 보기 (EDA) + Target 로그변환 필요 여부 확인
[Step3] 데이터 손질 (전처리: 삭제 / 결측치 / 이상치 / 인코딩 / 스케일링)
[Step4] AI 모델 학습 + 평가 (RMSE / RMSLE)
[Step5] 테스트 데이터 예측
[Step6] 제출 파일 만들기
[Step7] 성능 더 올리기 (선택)
```

## ★ 표시된 곳만 실제 문제에 맞게 바꾸면 됩니다!


---
## [Step1] 도구 꺼내기

```
요리 전에 냄비·칼·도마를 꺼내듯, 코딩 전에 라이브러리를 먼저 불러옵니다.
```

In [ ]:
# 필요하면 주석 해제 후 실행 (이미 설치되어 있으면 생략)
# pip install lightgbm xgboost


In [ ]:
# 기본 도구 (데이터 다루기)
import numpy as np
import pandas as pd

# 그래프 그리기
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')  # 경고 메시지 끄기


In [ ]:
# ★ 회귀 모델들 (이 중에서 가장 좋은 것을 골라 씁니다)
from sklearn.linear_model import LinearRegression, Ridge   # 선형회귀 계열
from sklearn.tree import DecisionTreeRegressor              # 결정 트리
from sklearn.ensemble import RandomForestRegressor          # 랜덤 포레스트
from sklearn.ensemble import ExtraTreesRegressor            # 엑스트라 트리
from sklearn.ensemble import VotingRegressor                # 앙상블(평균)
from xgboost import XGBRegressor                             # XGBoost
from lightgbm import LGBMRegressor                            # LightGBM

# 데이터 나누기 / 인코딩 / 스케일링 / 튜닝
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    KFold, cross_val_score
)
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer


### 데이터 불러오기

In [ ]:
train = pd.read_csv('./data/train.csv')  # 정답이 있는 학습용 데이터
test  = pd.read_csv('./data/test.csv')   # 정답 없는 예측용 데이터

print('train 크기:', train.shape)
print('test  크기:', test.shape)
train.head()


In [ ]:
train.info()


In [ ]:
test.info()


---
## [Step2] 데이터 보기 (EDA)

### (0) ★ 정답(Target) 컬럼 찾기 + 정답 없는 행 제거

```
train에는 있고 test에는 없는 컬럼이 보통 정답(Target)입니다.
정답이 비어있는(NaN) 행은 학습에 쓸 수 없으므로 미리 제거합니다.
```

In [ ]:
target_candidates = list(set(train.columns) - set(test.columns))
print('Target 후보:', target_candidates)

TARGET = target_candidates[0] if len(target_candidates) == 1 else 'shipping_costs'  # ★ 후보가 여러 개면 직접 지정
print('🎯 TARGET =', TARGET)

before = train.shape[0]
train = train.dropna(subset=[TARGET]).reset_index(drop=True)
print(f'정답 없는 행 제거: {before}행 → {train.shape[0]}행')

train[TARGET].describe()


### (1) Target 분포 확인 — 로그 변환이 필요한지 보기

```
왼쪽(원본)이 한쪽으로 심하게 치우쳐 있고 오른쪽(로그변환)이 더 종 모양에 가까우면
→ 로그 변환을 사용하세요! 평가지표가 RMSLE이면 반드시 로그 변환합니다.
```

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train[TARGET], ax=axes[0], kde=True).set_title('원본 분포')
sns.histplot(np.log1p(train[TARGET]), ax=axes[1], kde=True).set_title('log1p 변환 후')
plt.tight_layout()
plt.show()

print('왜도(skewness):', round(train[TARGET].skew(), 3), ' (0에 가까울수록 좌우대칭, 1 이상이면 로그변환 고려)')


### (2) 컬럼 처리 방향 자동 판단 — ❌삭제 / 🔤OHE / 🏷라벨인코딩 / 📊스케일링

```
❌ 삭제        → test에 없는 컬럼(누수) / ID류(고유값=행수) / 결측률 너무 높음
🔤 OHE        → 범주형인데 종류가 적음(≤10)
🏷 라벨인코딩  → 범주형인데 종류가 많음(11~50) — 트리 모델은 순서 상관없이 분기하므로 문제 없음
📊 스케일링    → 연속형 숫자
📅 날짜        → 자동으로 연/월/일/시로 쪼갠 뒤 다시 판단

★ 왜 OHE 대신 라벨인코딩을 쓰기도 하나요?
   범주가 50가지인 컬럼을 OHE 하면 컬럼이 50개 늘어나 학습이 느려지고
   차원의 저주(과적합 위험)가 생깁니다. 트리 계열 모델(RF/XGB/LGBM)은 숫자로 바꾸기만
   해도 기준값으로 잘 분기하므로 라벨인코딩으로 충분합니다.
```

In [ ]:
def analyze_columns(train_df, test_df, target_col,
                     ohe_max=10, label_max=50, null_ratio_drop=0.5):
    """
    train_df, test_df : 학습/테스트 데이터프레임 (test_df는 누수 판단에만 사용)
    target_col        : 정답 컬럼명
    반환값: drop_cols, ohe_cols, label_cols, num_cols
    """
    drop_cols, ohe_cols, label_cols, num_cols = [], [], [], []

    print(f"{'컬럼명':28} {'타입':10} {'고유값':>7} {'결측수':>7}  처리 방향")
    print('-' * 90)

    for col in train_df.columns:
        if col == target_col:
            print(f"{col:28} {'':10} {'':>7} {'':>7}  🎯 TARGET (건드리지 않음)")
            continue

        dtype = str(train_df[col].dtype)
        n_unique = train_df[col].nunique()
        n_null = train_df[col].isnull().sum()
        null_ratio = n_null / len(train_df)

        if test_df is not None and col not in test_df.columns:
            drop_cols.append(col)
            direction = '❌ 삭제 (test에 없는 컬럼 → 누수 위험)'
        elif n_unique >= len(train_df) * 0.98 and dtype != 'float64':
            drop_cols.append(col)
            direction = '❌ 삭제 (ID류, 고유값이 행 수만큼 많음)'
        elif null_ratio > null_ratio_drop:
            drop_cols.append(col)
            direction = f'❌ 삭제 (결측률 {null_ratio:.0%} — 채워도 신뢰하기 어려움)'
        elif 'datetime' in dtype:
            direction = '📅 날짜형 → 연/월/일/시로 분리 후 다시 판단 필요'
        # pandas 버전에 따라 문자열이 'object' 대신 'str'로 표시되기도 하므로
        # dtype 문자열 비교 대신 is_numeric_dtype()으로 판단 (버전 호환)
        elif pd.api.types.is_numeric_dtype(train_df[col]) and n_unique > ohe_max:
            num_cols.append(col)
            direction = '📊 스케일링 대상 (연속 숫자)'
        else:
            if n_unique <= ohe_max:
                ohe_cols.append(col)
                direction = f'🔤 OHE 추천 (고유값 {n_unique}개 ≤ {ohe_max})'
            elif n_unique <= label_max:
                label_cols.append(col)
                direction = f'🏷 라벨인코딩 추천 (고유값 {n_unique}개, OHE는 너무 많음)'
            else:
                drop_cols.append(col)
                direction = f'❌ 삭제 검토 (고유값 {n_unique}개 — 텍스트/식별자 가능성)'

        null_mark = f'  ⚠️ 결측 {n_null}개' if n_null > 0 else ''
        print(f"{col:28} {dtype:10} {n_unique:>7} {n_null:>7}  {direction}{null_mark}")

    print()
    print(f'✔️ 삭제 {len(drop_cols)}개 | OHE {len(ohe_cols)}개 | 라벨인코딩 {len(label_cols)}개 | 스케일링 {len(num_cols)}개')
    return drop_cols, ohe_cols, label_cols, num_cols


drop_cols, ohe_cols, label_cols, num_cols = analyze_columns(train, test, target_col=TARGET)
print('\ndrop_cols =', drop_cols)
print('ohe_cols  =', ohe_cols)
print('label_cols=', label_cols)
print('num_cols  =', num_cols)


### (3) 위 리스트가 마음에 안 들면 여기서 직접 조정하세요

In [ ]:
# ★ 필요하면 여기서 직접 수정
# 예) drop_cols.remove('vendor_id')
# 예) label_cols.append('vendor_id')

print('최종 drop_cols  :', drop_cols)
print('최종 ohe_cols   :', ohe_cols)
print('최종 label_cols :', label_cols)
print('최종 num_cols   :', num_cols)


### (4) 숫자형 변수 분포 확인 (히스토그램)

In [ ]:
show_cols = num_cols[:8]  # 너무 많으면 앞 8개만
fig, axes = plt.subplots(1, len(show_cols), figsize=(4*len(show_cols), 3))
if len(show_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, show_cols):
    sns.histplot(train[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()


### (5) 범주형 변수 분포 확인 (있을 때만)

In [ ]:
cat_show = (ohe_cols + label_cols)[:6]
if cat_show:
    fig, axes = plt.subplots(1, len(cat_show), figsize=(4*len(cat_show), 3))
    if len(cat_show) == 1:
        axes = [axes]
    for ax, col in zip(axes, cat_show):
        sns.countplot(x=train[col].astype(str), ax=ax)
        ax.set_title(col)
        ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('범주형 컬럼이 없습니다.')


---
## [Step3] 데이터 손질 (전처리)

### (1) 중복값 제거 — train만!

In [ ]:
print('중복 행 수:', train.duplicated().sum())
train = train.drop_duplicates().reset_index(drop=True)


### (2) 불필요한 컬럼 삭제 — analyze_columns의 drop_cols 적용

In [ ]:
train = train.drop(columns=drop_cols, errors='ignore')
test  = test.drop(columns=[c for c in drop_cols if c in test.columns], errors='ignore')
print('삭제 후 크기:', train.shape, test.shape)


### (3) 결측치(빈칸) 채우기 — 딱 한 번만!

```
📊 숫자형(num_cols)   → train의 중앙값(median)으로 채우기
🔤🏷 범주형(ohe/label) → train의 최빈값(mode)으로 채우기
⚠️ 항상 train 기준으로 계산한 값을 test에도 그대로 적용합니다!
```

In [ ]:
print('=== train 결측치 ===')
print(train.isnull().sum()[train.isnull().sum() > 0])
print('=== test  결측치 ===')
print(test.isnull().sum()[test.isnull().sum() > 0])


In [ ]:
# 📊 숫자형 → train 중앙값으로 채우기
for c in num_cols:
    fill_value = train[c].median()
    train[c] = train[c].fillna(fill_value)
    if c in test.columns:
        test[c] = test[c].fillna(fill_value)

# 🔤🏷 범주형 → train 최빈값으로 채우기
for c in ohe_cols + label_cols:
    mode_series = train[c].mode(dropna=True)
    fill_value = mode_series[0] if len(mode_series) > 0 else 'Unknown'
    train[c] = train[c].fillna(fill_value)
    if c in test.columns:
        test[c] = test[c].fillna(fill_value)

print('train 결측치 총합:', train.isnull().sum().sum())
print('test  결측치 총합:', test.isnull().sum().sum())


### (4) 아웃라이어(이상치) 제거 — train & Target 포함! (분류와 다른 점)

```
회귀는 Target(정답)도 이상치 제거 대상입니다.
왜? 정답이 대부분 400~1000인데 딱 하나가 80000이면 AI가 잘못된 패턴을 배웁니다.
IQR(사분위수 범위) 방법으로 자동으로 이상한 범위를 계산해서 제거합니다.
```

In [ ]:
plt.figure(figsize=(5, 3))
sns.boxplot(y=TARGET, data=train)
plt.title('Target 아웃라이어 확인')
plt.show()


In [ ]:
Q1, Q3 = train[TARGET].quantile(0.25), train[TARGET].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR

before = train.shape[0]
train = train[(train[TARGET] >= lower) & (train[TARGET] <= upper)].reset_index(drop=True)
print(f'Target 아웃라이어 제거: {before}행 → {train.shape[0]}행')


### (5) Feature(입력) / Target(정답) 분리

In [ ]:
Xtrain = train.drop(columns=[TARGET])
ytrain = train[TARGET]
Xtest  = test.copy()

print('Xtrain:', Xtrain.shape, ' ytrain:', ytrain.shape, ' Xtest:', Xtest.shape)


### (6) 인코딩 — 🔤 OHE + 🏷 라벨인코딩

```
⚠️ train/test를 따로따로 get_dummies 하면 서로 다른 컬럼이 생길 수 있습니다.
   그래서 train+test를 합쳐서 한 번에 인코딩한 뒤 다시 나눕니다.
```

In [ ]:
combined = pd.concat([Xtrain, Xtest], axis=0, keys=['train', 'test'])
combined = pd.get_dummies(combined, columns=ohe_cols)

for c in label_cols:
    le = LabelEncoder()
    combined[c] = le.fit_transform(combined[c].astype(str))

Xtrain_enc = combined.loc['train'].copy()
Xtest_enc  = combined.loc['test'].copy()

print('인코딩 후 컬럼 수 - Xtrain:', Xtrain_enc.shape[1], ' Xtest:', Xtest_enc.shape[1])
Xtrain_enc.head()


### (7) 스케일링 — 숫자 단위 통일

```
⚠️ 규칙: fit_transform(train) → train 기준 계산+변환 / transform(test) → 변환만!
```

In [ ]:
scaler = MinMaxScaler()

Xtrain_scaled = Xtrain_enc.copy()
Xtest_scaled  = Xtest_enc.copy()

if num_cols:
    Xtrain_scaled[num_cols] = scaler.fit_transform(Xtrain_enc[num_cols])
    Xtest_scaled[num_cols]  = scaler.transform(Xtest_enc[num_cols])

print('스케일링 완료!')
Xtrain_scaled.head()


### (8) Target 로그 변환 결정

```
Step2에서 본 분포가 한쪽으로 치우쳐 있었다면(왜도가 크다면) 로그 변환을 사용하세요.
평가지표가 RMSLE라면 무조건 사용합니다.
```

In [ ]:
USE_LOG = True  # ★ 로그 변환 여부 (Step2 분포를 보고 결정)

ytrain_target = np.log1p(ytrain) if USE_LOG else ytrain.copy()
print('사용할 target 샘플:', ytrain_target.values[:5])


---
## [Step4] AI 모델 학습 + 평가

### (1) 평가지표 함수 — RMSE / RMSLE

```
★ 문제에서 요구하는 평가지표를 EVAL_METRIC에 지정하세요.
```

In [ ]:
EVAL_METRIC = 'RMSLE'  # ★ 'RMSE' 또는 'RMSLE'

def rmse(actual, pred):
    return np.sqrt(mean_squared_error(actual, pred))

def rmsle(log_actual, log_pred):
    """둘 다 이미 log1p 변환된 값이라고 가정 (즉, RMSE와 같은 계산)"""
    return np.sqrt(mean_squared_error(log_actual, log_pred))

metric_func = rmsle if (EVAL_METRIC == 'RMSLE' and USE_LOG) else rmse
print(f'사용할 평가지표: {EVAL_METRIC} (로그변환={USE_LOG})')
# MAE, R²는 필요하면 sklearn 함수 그대로 사용: mean_absolute_error(y_val, pred), r2_score(y_val, pred)


### (2) train / val 분리

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    Xtrain_scaled, ytrain_target,
    test_size=0.2,
    random_state=42
)
print('학습용:', X_train.shape, ' 검증용:', X_val.shape)


### (3) 여러 모델 학습 → 가장 좋은 모델 찾기

```
★ 회귀는 지표가 낮을수록 좋습니다 (0에 가까울수록 정확).
```

In [ ]:
models = {
    'LinearRegression':      LinearRegression(),
    'Ridge':                 Ridge(random_state=42),
    'DecisionTreeRegressor': DecisionTreeRegressor(random_state=42),
    'RandomForestRegressor': RandomForestRegressor(random_state=42, n_jobs=-1),
    'ExtraTreesRegressor':   ExtraTreesRegressor(random_state=42, n_jobs=-1),
    'XGBRegressor':          XGBRegressor(random_state=42, n_jobs=-1),
    'LGBMRegressor':         LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
}

selected_estimator = None
selected_name = None
min_score = float('inf')

for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    pred = estimator.predict(X_val)
    score = metric_func(y_val, pred)
    print(f'[{name:24}] {EVAL_METRIC}: {score:.4f}')

    if score < min_score:
        selected_estimator, selected_name, min_score = estimator, name, score
        print('  ★ 현재 최선!')

print('=' * 60)
print('최종 선택 모델:', selected_name, f' ({EVAL_METRIC}:', round(min_score, 4), ')')

PARAM_GUIDE = {
    'LGBMRegressor':         {'learning_rate': [0.05, 0.1, 0.14], 'n_estimators': [100, 200, 300]},
    'XGBRegressor':          {'learning_rate': [0.05, 0.1, 0.2],  'n_estimators': [100, 200, 300], 'max_depth': [3, 5, 7]},
    'RandomForestRegressor': {'n_estimators': [100, 200, 300], 'max_depth': [None, 5, 10]},
    'ExtraTreesRegressor':   {'n_estimators': [100, 200, 300], 'max_depth': [None, 5, 10]},
    'DecisionTreeRegressor': {'max_depth': [3, 5, 7, 10, None], 'min_samples_split': [2, 5, 10]},
    'Ridge':                 {'alpha': [0.1, 1.0, 10.0]},
}
if selected_name in PARAM_GUIDE:
    print(f'\n[{selected_name}] 아래 파라미터를 GridSearchCV에 복붙하세요:')
    print('parameters =', PARAM_GUIDE[selected_name])


### (4) 하이퍼파라미터 튜닝 (GridSearchCV)

In [ ]:
# ★ 위에서 출력된 파라미터를 여기에 복붙!
parameters = PARAM_GUIDE.get(selected_name, {})

grid_reg = GridSearchCV(
    selected_estimator,
    param_grid=parameters,
    cv=5,
    scoring=make_scorer(metric_func, greater_is_better=False),
    n_jobs=-1,
)
grid_reg.fit(Xtrain_scaled, ytrain_target)

best_model = grid_reg.best_estimator_
print('최적 파라미터:', grid_reg.best_params_)
print(f'최적 CV 성능 ({EVAL_METRIC}):', abs(round(grid_reg.best_score_, 4)))


---
## [Step5] 테스트 데이터 예측

In [ ]:
pred_test = best_model.predict(Xtest_scaled)

if USE_LOG:
    pred_test = np.expm1(pred_test)      # 로그 역변환
    pred_test = np.clip(pred_test, 0, None)  # 음수 예측 방지

print('예측값 샘플:', pred_test[:5])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(ytrain,    ax=axes[0]).set_title('Train 실제값 분포')
sns.histplot(pred_test, ax=axes[1]).set_title('Test 예측값 분포')
plt.tight_layout()
plt.show()


---
## [Step6] 제출 파일 만들기

In [ ]:
submission = pd.read_csv('./data/sample-submission.csv')
submission[TARGET] = pred_test
submission.to_csv('./data/MySubmission.csv', index=False)

print('✅ 제출 파일 저장 완료! → data/MySubmission.csv')
submission.head()


---
## [Step7] 성능 더 올리기 (선택)

```
아래는 시간이 남을 때 시도해볼 수 있는 개선 방법들입니다.
전부 다 할 필요는 없고, 필요한 것만 골라 쓰세요.
```

### (1) K-Fold 교차검증 — 더 믿을 만한 성능 측정

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    best_model, Xtrain_scaled, ytrain_target, cv=kf,
    scoring=make_scorer(metric_func, greater_is_better=False), n_jobs=-1
)
cv_scores = np.abs(cv_scores)
print(f'5-Fold {EVAL_METRIC} 점수들:', np.round(cv_scores, 4))
print('평균:', round(cv_scores.mean(), 4), ' 표준편차:', round(cv_scores.std(), 4))


### (2) Early Stopping — 과적합 방지 + 튜닝 시간 단축 (XGBoost/LightGBM)

In [ ]:
early_model = LGBMRegressor(
    n_estimators=1000,   # 넉넉하게 크게 주고
    learning_rate=0.05,
    random_state=42,
    verbose=-1,
)
early_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[],  # 버전에 따라 lightgbm.early_stopping(30) 콜백을 넣어 조기 종료할 수 있습니다
)
print('Early stopping은 n_estimators를 크게 잡아도 자동으로 적당한 지점에서 멈추게 해줍니다.')
print('(LightGBM 버전에 따라 import lightgbm; callbacks=[lightgbm.early_stopping(30)] 형태로 사용)')


### (3) 파생변수(피처 엔지니어링) 아이디어

In [ ]:
# 예시: 두 숫자 컬럼의 비율/차이로 새로운 의미 있는 변수 만들기
# Xtrain_scaled['새컬럼'] = Xtrain['A컬럼'] / (Xtrain['B컬럼'] + 1e-6)
# Xtest_scaled['새컬럼']  = Xtest['A컬럼']  / (Xtest['B컬럼']  + 1e-6)

# 예시: 위도/경도가 있으면 두 지점 사이 직선 거리 계산
# dist_lon = Xtrain['pickup_lon'] - Xtrain['dropoff_lon']
# dist_lat = Xtrain['pickup_lat'] - Xtrain['dropoff_lat']
# Xtrain_scaled['distance'] = np.sqrt(dist_lon**2 + dist_lat**2)

print('데이터 도메인 지식이 있다면 위 방식으로 파생변수를 만들어 재학습해보세요.')


### (4) 탐색 범위 넓히기 — RandomizedSearchCV (조합이 너무 많을 때)

In [ ]:
wide_parameters = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 5, 7, 10, 15],
}

random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=wide_parameters,
    n_iter=10,          # 전체 조합 중 10개만 무작위로 시도 → GridSearch보다 빠름
    cv=5,
    scoring=make_scorer(metric_func, greater_is_better=False),
    random_state=42,
    n_jobs=-1,
)
random_search.fit(Xtrain_scaled, ytrain_target)
print('RandomizedSearch 최적 파라미터:', random_search.best_params_)
print(f'RandomizedSearch 최적 성능 ({EVAL_METRIC}):', abs(round(random_search.best_score_, 4)))


### (5) 피처 중요도 기반 가지치기

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    order = np.argsort(importances)[::-1]

    plt.figure(figsize=(8, 4))
    top_n = min(15, len(order))
    sns.barplot(x=importances[order[:top_n]], y=Xtrain_scaled.columns[order[:top_n]])
    plt.title('Top Feature Importances')
    plt.tight_layout()
    plt.show()

    low_importance_cols = Xtrain_scaled.columns[order[top_n:]]
    print(f'중요도 하위 {len(low_importance_cols)}개 컬럼은 제거 후 재학습을 시도해볼 수 있습니다.')
else:
    print('이 모델은 feature_importances_를 지원하지 않습니다 (예: LinearRegression, Ridge).')


### (6) 간단 앙상블 — 상위 모델들의 예측 평균

In [ ]:
voting_model = VotingRegressor(
    estimators=[
        ('rf', RandomForestRegressor(random_state=42, n_jobs=-1)),
        ('xgb', XGBRegressor(random_state=42, n_jobs=-1)),
        ('lgbm', LGBMRegressor(random_state=42, verbose=-1)),
    ]
)
voting_model.fit(X_train, y_train)
voting_pred = voting_model.predict(X_val)
print(f'Voting Ensemble {EVAL_METRIC}:', round(metric_func(y_val, voting_pred), 4))
print(f'단일 최선 모델 {EVAL_METRIC}  :', round(min_score, 4))
print('→ 앙상블 점수가 더 낮으면(좋으면) 최종 제출을 voting_model 예측으로 바꿔도 좋습니다.')


---
## 🚨 자주 하는 실수 체크리스트

```
1. test에 fit_transform() 사용                        ❌ → transform()만 사용
2. Target 컬럼을 feature(X)에 포함                     ❌ → drop([TARGET]) 먼저!
3. train/test를 따로 get_dummies → 컬럼 수가 달라짐    ❌ → 합쳐서 인코딩 후 분리
4. 로그 변환 후 예측했는데 expm1() 역변환 안 함         ❌ → 반드시 역변환!
5. train 따로 test 따로 결측치 기준값 계산              ❌ → 항상 train 기준값을 test에도 적용
6. Target 아웃라이어를 안 지움(회귀는 분류와 다름)      ❌ → IQR로 Target도 정리
```
